# 08 (Bonus) - Agentic Browsing with Jina Reader

This is the **exact same mechanism** your Telegram bot already uses -- `read_dynamic_webpage` in `agent.py` calls the same free reader service this notebook does. No new infrastructure, no new dependencies, pure Python.

What we'll add here: instead of reading *one* page, the agent will **decide for itself which link to follow** to reach an answer that isn't on the page it starts on -- real multi-step navigation, without a real browser, MCP, or anything beyond the tool-calling loop from notebook 03.


In [ ]:
%pip install -q openai-agents httpx

from agents import set_default_openai_key

# Don't share or commit this notebook with your key filled in.
OPENAI_API_KEY = "sk-..."  # <-- paste your key here
set_default_openai_key(OPENAI_API_KEY)

print("Ready to go.")

## A tool that reads a page *and* lists where it could go next

`read_dynamic_webpage` in the capstone repo just returns page text. Here, we add one thing: the `X-With-Links-Summary` header, which asks Jina Reader to append a clean list of every link on the page. That link list is what makes navigation possible -- without it, the agent has no way to know what pages exist beyond the one it's looking at.

We keep both the page content *and* the links list, but cap each separately -- a blind `[:4000]` truncation (like the simpler version in `agent.py`) risks cutting off the links entirely if they happen to load after a wall of page content.


In [ ]:
import httpx


def fetch_page(url: str) -> str:
    resp = httpx.get(
        f"https://r.jina.ai/{url}",
        headers={"X-No-Cache": "true", "X-With-Links-Summary": "true"},
        timeout=30,
    )
    text = resp.text

    if "Links/Buttons:" in text:
        content, links = text.split("Links/Buttons:", 1)
        return content[:2000] + "\n\nLinks/Buttons:" + links[:3000]
    return text[:3000]

## Try a single fetch


In [ ]:
print(fetch_page('https://books.toscrape.com/'))

Notice the output ends with a `Links/Buttons:` section -- clean category links like `Mystery`, `Poetry`, `Travel`. That's what the agent will use to decide where to go.

## Turning it into a tool

Same idea as every other tool in this course -- wrap the function with `@function_tool` and give it a docstring that tells the agent *when* and *why* to call it again with a different URL.


In [ ]:
from agents import function_tool


@function_tool
def read_webpage(url: str) -> str:
    """Fetch a webpage's content, plus a list of links found on it.

    Use this to read a page, and again with a different URL (one you found
    in that page's link list) if the answer turns out to be on another page.

    Args:
        url: The full URL of the page to read.
    """
    return fetch_page(url)

## Making it agentic

There's no special "navigation" code to write. The agent already knows how to call a tool more than once in a run (notebook 03) -- all it needs is a reason to. Give it a task that **isn't answered on the first page**, and clear instructions to keep reading if it isn't, and it'll pick a link from the list and fetch that page too.


In [ ]:
from agents import Agent, Runner

agent = Agent(
    name="Web Navigator",
    instructions="""
    Use read_webpage to answer questions about live websites.
    If the answer isn't on the page you're given, look at that page's
    Links/Buttons list, pick the link most likely to have the answer, and
    call read_webpage again with that URL. Keep going until you find it.
    """,
    tools=[read_webpage],
)

result = await Runner.run(
    agent,
    "Start at https://books.toscrape.com/ -- how many books are listed in the Mystery category?",
)
print(result.final_output)

## Watching it navigate

Reusing `print_trace` from notebook 03 makes the decision-making visible -- you should see it call `read_webpage` on the homepage, then call it *again* with the Mystery category URL it picked from the links list.


In [ ]:
def print_trace(result):
    for item in result.new_items:
        if item.type == "tool_call_item":
            name = item.tool_name or "tool"
            args = getattr(item.raw_item, "arguments", "")
            print(f"🔧 {name}({args})")
        elif item.type == "tool_call_output_item":
            preview = str(item.output)[:120]
            print(f"   -> {preview}...")

    print(f"\n💬 {result.final_output}")


result = await Runner.run(
    agent,
    "Start at https://books.toscrape.com/ -- what's the cheapest book in the Poetry category?",
)
print_trace(result)

You should see two (or more) `🔧 read_webpage(...)` lines with *different* URLs -- the second one chosen by the agent itself from the first page's link list, not hardcoded anywhere in this notebook.

### Exercise

Try a task that needs **three** hops instead of two -- e.g. starting from the homepage, find a specific book's price inside a category page (homepage -> category -> book detail page). Watch the trace to confirm it actually took three steps.


In [ ]:
# TODO: try a 3-hop task, e.g.
# result = await Runner.run(agent, "Start at https://books.toscrape.com/ -- what is the price of the book 'Sharp Objects'?")
# print_trace(result)


## Recap

| What you did | Where it lives in the capstone repo |
|---|---|
| Fetching a JS-rendered page via Jina Reader | `agent.py` -- `read_dynamic_webpage` (same service, same idea) |
| Multi-hop navigation | Nothing new -- it's the same tool-calling loop from notebook 03, just with a tool whose output includes more places to go |

One real difference: `read_dynamic_webpage` in the deployed bot doesn't send `X-With-Links-Summary`, so it doesn't get a clean link list back -- meaning the deployed bot's browsing is effectively single-hop today. Adding that header (and the two-part truncation from this notebook) would be a genuine, small upgrade if you wanted the deployed bot to navigate multi-hop too -- not something this notebook changes for you, just worth knowing it's the one real gap between what you built here and what's live.

**Next:** open `07_from_notebook_to_telegram_bot.ipynb` if you haven't already, or head back to the capstone repo's `README.md`.
